In [43]:
import numpy as np
import pandas as pd

In [44]:
biomarkers = ['KLRD1','FCGR1A','NKG7','CXCL11','CXCL9','PRF1','MSR1',
'EOMES','APOL3','SAMD3','IRF8','GBP1','HLA-F','RUNX3','STAT1','TRAF5','CSF2RA',
'CCL5','HCP5','ENTPD1']

cell_cycle = ['ANLN','ASPM','BUB1','BUB1B','CCNA2','CCNB1','CDKN3','CENPA','CENPF','CEP55','DLGAP5','DTL','EZH2','KIAA0101','KIF11','KIF20A','MKI67','NUSAP1','RRM2','TOP2A','TPX2','TTK','TYMS','UBE2C']

In [45]:
degs_liver = pd.read_excel("./data/degs_result_liver.xlsx")
degs_liver.index = degs_liver['Gene']
degs_lung = pd.read_excel("./data/degs_result_lung.xlsx")
degs_lung.index = degs_lung['Gene']
degs_heart = pd.read_excel("./data/degs_result_heart.xlsx")
degs_heart.index = degs_heart['Gene']
degs_kidney = pd.read_excel("./data/degs_result_kidney.xlsx")
degs_kidney.index  = degs_kidney['Gene']

In [46]:
# List of all your result dataframes
deg_tables = {
    'liver': degs_liver,
    'lung': degs_lung,
    'heart': degs_heart,
    'kidney': degs_kidney
}

In [47]:
biomarker_subsets = {}
cell_cycle_subsets = {}

for organ, df in deg_tables.items():    
    existing_biomarkers = df.index.intersection(biomarkers)
    biomarker_subsets[organ] = df.loc[existing_biomarkers]
    
    existing_cc = df.index.intersection(cell_cycle)
    cell_cycle_subsets[organ] = df.loc[existing_cc]
    
    print(f"{organ.capitalize()}: Found {len(existing_biomarkers)} biomarkers and {len(existing_cc)} cell cycle genes.")


Liver: Found 20 biomarkers and 24 cell cycle genes.
Lung: Found 20 biomarkers and 24 cell cycle genes.
Heart: Found 20 biomarkers and 24 cell cycle genes.
Kidney: Found 20 biomarkers and 24 cell cycle genes.


In [48]:
organs = ['liver', 'kidney', 'heart', 'lung']

biomarker_list = []
for organ in organs:
    df = biomarker_subsets[organ][['logFC', 'adj.P.Val']].copy()
    df.columns = [f'{organ}.logFC', f'{organ}.adj.P.Val']
    biomarker_list.append(df)
combined_biomarkers = pd.concat(biomarker_list, axis=1)
combined_biomarkers.insert(0, 'Gene', combined_biomarkers.index)

cc_list = []
for organ in organs:
    df = cell_cycle_subsets[organ][['logFC', 'adj.P.Val']].copy()
    df.columns = [f'{organ}.logFC', f'{organ}.adj.P.Val']
    cc_list.append(df)
combined_cell_cycle = pd.concat(cc_list, axis=1)
combined_cell_cycle.insert(0, 'Gene', combined_cell_cycle.index)

In [52]:
def get_status(logFC, adj_P):
    if adj_P >= 0.05: return "non-significant"
    return "upregulated" if logFC > 0 else "downregulated"

for organ in organs:
    lfc, pval, stat = f'{organ}.logFC', f'{organ}.adj.P.Val', f'{organ}.status'
    
    combined_biomarkers[stat] = combined_biomarkers.apply(lambda x: get_status(x[lfc], x[pval]), axis=1)
    combined_cell_cycle[stat] = combined_cell_cycle.apply(lambda x: get_status(x[lfc], x[pval]), axis=1)

In [53]:
reordered_columns = [
    'Gene',
    'liver.logFC', 'liver.adj.P.Val', 'liver.status',
    'kidney.logFC', 'kidney.adj.P.Val', 'kidney.status',
    'heart.logFC', 'heart.adj.P.Val', 'heart.status',
    'lung.logFC', 'lung.adj.P.Val', 'lung.status'
]

combined_biomarkers = combined_biomarkers[reordered_columns]
combined_cell_cycle = combined_cell_cycle[reordered_columns]

combined_biomarkers.to_excel("./data/biomarker_final_report.xlsx", index=False)
combined_cell_cycle.to_excel("./data/cell_cycle_final_report.xlsx", index=False)